In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
.master("local[*]") \
.appName('test') \
.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 16:26:49 WARN Utils: Your hostname, Translite, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/08 16:26:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 16:26:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/08 16:26:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
df_yellow_2025 = spark.read.parquet("data/pq/yellow/2025/11")

In [4]:
df_yellow_2025.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [7]:
# Renaming the columns for the ease of use
df_yellow_2025 = df_yellow_2025.withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [11]:
df_yellow_2025.registerTempTable('yellow_2025')

In [21]:
# Question 3 - How many taxi trips were there on the 15th of November?
spark.sql("""
SELECT COUNT(*) FROM yellow_2025
WHERE date_format(pickup_datetime, 'yyyy-MM-dd') = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [28]:
# Question 4 - What is the length of the longest trip in the dataset in hours?
spark.sql("""
  SELECT MAX((unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 3600.0 ) AS max_hours
  FROM yellow_2025
  WHERE pickup_datetime IS NOT NULL AND dropoff_datetime IS NOT NULL
""").show()

+---------+
|max_hours|
+---------+
|90.646667|
+---------+



In [29]:
# Downloading zone lookup data:
zone_pickup = spark.read \
.option("header", "true") \
.csv("data/taxi_zone_lookup.csv")
zone_pickup.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [30]:
zone_pickup.registerTempTable('zone_pickup')

In [45]:
# Question 6 - least frequent pickup location zone
spark.sql("""
SELECT y.PULocationID,
       z.Zone,
       COUNT(*) AS pickups
FROM yellow_2025 y
JOIN zone_pickup z
  ON y.PULocationID = z.LocationID
GROUP BY y.PULocationID, z.Zone
ORDER BY pickups ASC
""").show()

+------------+--------------------+-------+
|PULocationID|                Zone|pickups|
+------------+--------------------+-------+
|          84|Eltingville/Annad...|      1|
|         105|Governor's Island...|      1|
|           5|       Arden Heights|      1|
|         187|       Port Richmond|      3|
|         199|       Rikers Island|      4|
|         111| Green-Wood Cemetery|      4|
|         204|   Rossville/Woodrow|      4|
|         109|         Great Kills|      4|
|           2|         Jamaica Bay|      5|
|         251|         Westerleigh|     12|
|         176|             Oakwood|     14|
|         245|       West Brighton|     14|
|         172|New Dorp/Midland ...|     14|
|          59|        Crotona Park|     14|
|         253|       Willets Point|     15|
|          27|Breezy Point/Fort...|     16|
|         206|Saint George/New ...|     17|
|          30|       Broad Channel|     18|
|         156|     Mariners Harbor|     21|
|         118|Heartland Village.

In [46]:
print(f"Spark version: {spark.version}")

Spark version: 4.1.1
